## 1. Setup - Import thư viện và cấu hình

In [50]:
import os
import warnings
import sys
# from pathlib import Path
from collections import defaultdict
import time
import logging


# LangChain imports
from langchain_community.document_loaders import UnstructuredPDFLoader
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document

# Docling imports
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions, TableFormerMode

import chromadb

# Tắt warnings
warnings.filterwarnings('ignore')

print("✅ Đã import thành công tất cả thư viện!")

✅ Đã import thành công tất cả thư viện!


In [51]:
import io
from contextlib import redirect_stdout, redirect_stderr

# Thiết lập root logger ở mức WARNING để giảm hết INFO/DEBUG
logging.basicConfig(level=logging.WARNING, handlers=[logging.StreamHandler(sys.stdout)])

# Giảm mức logging cho các logger hay in nhiều thông báo
NOISY_LOGGERS = [
    "RapidOCR", "rapidocr", "onnxruntime", "chromadb", "urllib3",
    "numba", "transformers", "PIL", "google", 
    "docling", "docling_core", "docling_parse",  # <-- THÊM CÁC LOGGER CỦA DOCLING
    "easyocr", "torchvision"  # Thêm nếu dùng EasyOCR
]

for name in NOISY_LOGGERS:
    logging.getLogger(name).setLevel(logging.ERROR)  # <-- Chỉ hiện ERROR trở lên

# Tắt propagation để các logger con không in ra root
for name in NOISY_LOGGERS:
    logger = logging.getLogger(name)
    logger.propagate = False

# Một biến helper để tạm chuyển hướng stdout/stderr (dùng khi 1 call cụ thể in quá nhiều)
def _suppress_io():
    return redirect_stdout(io.StringIO()), redirect_stderr(io.StringIO())

print("✅ Đã cấu hình logging để ẩn INFO/DEBUG từ thư viện bên ngoài")

✅ Đã cấu hình logging để ẩn INFO/DEBUG từ thư viện bên ngoài


In [52]:
# Cấu hình đường dẫn cho Tesseract và Poppler (cần cho OCR)
tesseract_dir = r"C:\Program Files\Tesseract-OCR"
poppler_dir = r"C:\poppler-25.07.0\Library\bin"

current_path = os.environ.get("PATH", "")
if tesseract_dir not in current_path:
    os.environ["PATH"] = current_path + os.pathsep + tesseract_dir
if poppler_dir not in current_path:
    os.environ["PATH"] = os.environ.get("PATH", "") + os.pathsep + poppler_dir

os.environ["TESSDATA_PREFIX"] = r"C:\Program Files\Tesseract-OCR\tessdata\\"

print("✅ Đã cấu hình PATH cho Tesseract và Poppler")

✅ Đã cấu hình PATH cho Tesseract và Poppler


## 2. Khởi tạo Embedding Model và ChromaDB

In [53]:
print("⏳ Đang khởi tạo mô hình embedding...")

# Khởi tạo embedding model (hỗ trợ đa ngôn ngữ, tốt cho tiếng Việt)
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

print("✅ Mô hình embedding đã sẵn sàng!")
print(f"📌 Model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
print(f"🌍 Hỗ trợ: Tiếng Việt và 50+ ngôn ngữ khác")

⏳ Đang khởi tạo mô hình embedding...


2025-12-11 10:14:34,529 - INFO - Load pretrained SentenceTransformer: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


✅ Mô hình embedding đã sẵn sàng!
📌 Model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
🌍 Hỗ trợ: Tiếng Việt và 50+ ngôn ngữ khác


In [54]:
## Tạo ChromaDB-compatible embedding function
from chromadb.api.types import EmbeddingFunction
from typing import List

class HuggingFaceEmbeddingFunction(EmbeddingFunction):
    """
    Wrapper để HuggingFaceEmbeddings tương thích với ChromaDB
    """
    def __init__(self, hf_embeddings):
        self.hf_embeddings = hf_embeddings
    
    def __call__(self, input: List[str]) -> List[List[float]]:
        """
        ChromaDB yêu cầu signature này: __call__(self, input: List[str])
        """
        return self.hf_embeddings.embed_documents(input)

# Tạo embedding function tương thích với ChromaDB
chroma_embedding_function = HuggingFaceEmbeddingFunction(embedding_model)

print("✅ Đã tạo ChromaDB-compatible embedding function")

✅ Đã tạo ChromaDB-compatible embedding function


In [55]:
# Cấu hình ChromaDB
db_path = "./chroma_db"
collection_name_unstructured = "raw_unstructured_collection"
collection_name_docling = "raw_docling_collection"

print(f"📦 ChromaDB persist directory: {os.path.abspath(db_path)}")
print(f"📚 Collection 1 (Unstructured): {collection_name_unstructured}")
print(f"📚 Collection 2 (Docling): {collection_name_docling}")

📦 ChromaDB persist directory: d:\LearnUniversity\Semester11\TieuLuanChuyenNganh\LLM_University_Policy_Chatbot\chroma_db
📚 Collection 1 (Unstructured): raw_unstructured_collection
📚 Collection 2 (Docling): raw_docling_collection


## 3. Helper Function 
### 3.1. Xử lý PDF bằng Unstructured (mode="elements")

**Logic:**
- Dùng `mode="elements"` để lấy từng element với metadata `page_number`
- Nhóm (group) các elements theo số trang
- Gộp text của tất cả elements trong cùng 1 trang thành 1 Document

In [56]:
def process_with_unstructured(file_path: str) -> list[Document]:
    """
    Xử lý PDF bằng UnstructuredPDFLoader với mode="elements".
    Tự động fallback từ hi_res → ocr_only nếu hi_res fail.
    Nhóm các elements theo page_number và tạo 1 Document cho mỗi trang.
    
    Args:
        file_path: Đường dẫn đến file PDF
        
    Returns:
        List of Document objects, mỗi Document tương ứng 1 trang
    """
    print(f"  📄 [Unstructured] Đang xử lý: {os.path.basename(file_path)}...")
    
    # Thử hi_res trước (strategy tốt nhất)
    strategies_to_try = ["hi_res", "ocr_only"]
    for strategy in strategies_to_try:
        try:
            if strategy != "hi_res":
                print(f"    ⚠️ Fallback sang strategy='{strategy}'...")
            # Khởi tạo loader với mode="elements" để lấy từng element riêng biệt với strategy động
            loader = UnstructuredPDFLoader(
                file_path,
                mode="elements",  # QUAN TRỌNG: lấy từng element với metadata page_number
                strategy=strategy,  # Độ phân giải cao cho OCR
                languages=["vie"],  # Tiếng Việt
                pdf_infer_table_structure=True
            )
            
            # Load tất cả elements
            elements = loader.load()

           # Nhóm elements theo page_number (không loại bỏ hoàn toàn page thiếu)
            pages_dict = defaultdict(list)
            for element in elements:
                page_num = element.metadata.get('page_number')
                if page_num is None:
                    continue
                content = element.page_content
                if content not in [None, "None", ""]:
                    pages_dict[page_num].append(content)

            # Xác định số trang theo max(page_number) từ elements
            max_page = max(pages_dict.keys()) if pages_dict else 1
            
            # Tạo Document cho từng trang
            documents = []
            for page_num in range(1, max_page + 1):
                # Gộp tất cả text của các elements trong cùng 1 trang
                combined_text = "\n\n".join(pages_dict.get(page_num, []))  # "" nếu không có
                
                # Tạo Document với metadata rõ ràng
                doc = Document(
                    page_content=combined_text,
                    metadata={
                        'source': file_path,
                        'filename': os.path.basename(file_path),
                        'page': page_num,
                        'method': 'unstructured',
                    }
                )
                documents.append(doc)
            
            print(f"    ✅ Hoàn tất với strategy='{strategy}'! Tổng {len(documents)} trang")
            return documents
            
        except Exception as e:
            print(f"    ⚠️ Strategy '{strategy}' failed: {e}")
            # Nếu không phải strategy cuối cùng, thử strategy tiếp theo
            if strategy != strategies_to_try[-1]:
                continue
            else:
                # Đã thử hết strategies, return rỗng
                print(f"    ❌ ĐÃ THỬ HẾT {len(strategies_to_try)} strategies, không xử lý được")
                return []  # Trả về list rỗng nếu có lỗi

print("✅ Đã định nghĩa hàm process_with_unstructured() với auto-fallback hi_res → ocr_only")

✅ Đã định nghĩa hàm process_with_unstructured() với auto-fallback hi_res → ocr_only


### 3.2. Xử lý PDF bằng Docling (theo từng trang)

**Logic:**
- Dùng `DocumentConverter` trực tiếp (KHÔNG dùng DoclingLoader vì nó tự chunk)
- Cấu hình OCR gốc chứ không dùng tiếng Việt (EasyOCR) để giữ layout
- Lặp qua từng trang trong `result.document.pages`
- Export mỗi trang ra Markdown

In [57]:
def process_with_docling(file_path: str) -> list[Document]:
    """
    Trích xuất mỗi trang PDF thành 1 Document bằng Docling, sử dụng export_to_markdown(page_no=...).
    """
    print(f"  📄 [Docling] Đang xử lý: {os.path.basename(file_path)}...")
    try:
        # Cấu hình pipeline
        pipeline_options = PdfPipelineOptions()
        pipeline_options.do_ocr = True
        pipeline_options.do_table_structure = True
        pipeline_options.table_structure_options.mode = TableFormerMode.ACCURATE

        # Tạo converter
        converter = DocumentConverter(
            format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)}
        )

        # Convert PDF
        result = converter.convert(file_path)
        doc_obj = result.document

        # Kiểm tra nếu không có pages
        if not hasattr(doc_obj, "pages") or not doc_obj.pages:
            print("    ⚠️ Không có thuộc tính pages. Trích xuất toàn bộ.")
            whole_md = doc_obj.export_to_markdown() if hasattr(doc_obj, "export_to_markdown") else str(doc_obj)
            return [
                Document(
                    page_content=whole_md,
                    metadata={
                        "source": file_path,
                        "filename": os.path.basename(file_path),
                        "page": 1,
                        "method": "docling",
                    },
                )
            ]

        # Trích xuất nội dung từng trang
        documents = []
        for page_no in range(1, len(doc_obj.pages) + 1):
            page_md = doc_obj.export_to_markdown(page_no=page_no)
            if page_md.strip():  # Bỏ qua trang rỗng
                documents.append(
                    Document(
                        page_content=page_md.strip(),
                        metadata={
                            "source": file_path,
                            "filename": os.path.basename(file_path),
                            "page": page_no,
                            "method": "docling",
                        },
                    )
                )

        print(f"    ✅ Hoàn tất! Tổng {len(documents)} trang Docling")
        return documents

    except Exception as e:
        print(f"    ❌ LỖI Docling: {e}")
        import traceback
        traceback.print_exc()
        return []

print("✅ Định nghĩa hàm process_with_docling()")

✅ Định nghĩa hàm process_with_docling()


### 3.3. Lưu Documents vào ChromaDB

In [58]:
def save_to_chromadb(documents: list[Document], collection_name: str, filename: str):
    """
    Lưu documents vào ChromaDB collection với UPSERT mode (ghi đè nếu ID trùng).
    
    Args:
        documents: List các Document cần lưu
        collection_name: Tên collection
        filename: Tên file (dùng để tạo ID deterministc)
    """
    if not documents:
        print(f"    ⚠️ Không có documents để lưu vào {collection_name}")
        return
    
    try:
        # Kết nối đến ChromaDB
        client = chromadb.PersistentClient(path=db_path)
        
        # Lấy hoặc tạo collection
        collection = client.get_or_create_collection(
            name=collection_name,
            embedding_function=chroma_embedding_function
        )
        
        # Tạo IDs deterministc từ filename + page
        doc_ids = []
        docs_content = []
        docs_metadata = []
        
        for doc in documents:
            page = doc.metadata.get('page', 1)
            # ID: filename_page_X
            doc_id = f"{filename}_page_{page}"
            doc_ids.append(doc_id)
            docs_content.append(doc.page_content)
            docs_metadata.append(doc.metadata)
        
        # UPSERT: ghi đè nếu ID trùng
        collection.upsert(
            ids=doc_ids,
            documents=docs_content,
            metadatas=docs_metadata
        )
        
        print(f"    💾 Đã lưu/cập nhật {len(documents)} documents vào collection '{collection_name}'")
        
    except Exception as e:
        print(f"    ❌ LỖI khi lưu vào ChromaDB: {e}")

print("✅ Đã cập nhật hàm save_to_chromadb() dùng UPSERT")

✅ Đã cập nhật hàm save_to_chromadb() dùng UPSERT


### 3.4. Kiểm tra File đã xử lý chưa

In [59]:
def is_file_processed(collection, filename: str) -> bool:
    """
    Kiểm tra xem file đã được xử lý chưa (có document nào của file này trong collection không).
    
    Args:
        collection: ChromaDB collection object
        filename: Tên file cần kiểm tra
        
    Returns:
        bool: True nếu file đã có ít nhất 1 document, False nếu chưa
    """
    try:
        results = collection.get(
            where={"filename": {"$eq": filename}},
            limit=1
        )
        return len(results['ids']) > 0
    except Exception as e:
        print(f"    ⚠️ Lỗi khi kiểm tra file: {e}")
        return False  # Nếu lỗi thì coi như chưa xử lý để an toàn

print("✅ Đã định nghĩa hàm is_file_processed()")

✅ Đã định nghĩa hàm is_file_processed()


In [60]:
# # Code xóa collection để làm mới lại
# print("🗑️ Đang xóa collection refined_pages_collection...")
# try:
#     client.delete_collection(name=collection_name_unstructured)
#     print("✅ Đã xóa collection cũ thành công")
# except Exception as e:
#     print(f"ℹ️ Collection không tồn tại hoặc đã bị xóa: {e}")

# # Cập nhật lại existing_collections
# existing_collections = {col.name for col in client.list_collections()}
# print(f"📋 Collections còn lại: {existing_collections}")

## 4. Kiểm tra collections đã tồn tại

In [61]:
print("\n🔍 KIỂM TRA COLLECTIONS HIỆN TẠI")
print("="*100)

# Khởi tạo ChromaDB client
client = chromadb.PersistentClient(path=db_path)

# Lấy danh sách collections hiện có
existing_collections = {col.name for col in client.list_collections()}

print(f"📋 Collections hiện có: {existing_collections}")

# Tạo collections nếu chưa tồn tại
if collection_name_unstructured not in existing_collections:
    print(f"🆕 Tạo collection mới: {collection_name_unstructured}")
if collection_name_docling not in existing_collections:
    print(f"🆕 Tạo collection mới: {collection_name_docling}")

print("\n✅ Sẵn sàng xử lý incremental (chỉ file mới)\n")


🔍 KIỂM TRA COLLECTIONS HIỆN TẠI
📋 Collections hiện có: {'academic_regulations', 'academic_regulations_test', 'raw_unstructured_collection', 'refined_pages_collection', 'raw_docling_collection'}

✅ Sẵn sàng xử lý incremental (chỉ file mới)



## 5. Main Execution Loop - Xử lý tất cả PDF files

In [62]:
# Đường dẫn thư mục data
data_dir = "./data"

# Lấy danh sách tất cả file PDF
if not os.path.exists(data_dir):
    raise FileNotFoundError(f"❌ Không tìm thấy thư mục: {data_dir}")

pdf_files = [f for f in os.listdir(data_dir) if f.lower().endswith(".pdf")]

print("="*100)
print(f"📁 Tìm thấy {len(pdf_files)} file PDF trong thư mục {data_dir}")
print("="*100)

# Kết nối đến collections để kiểm tra
try:
    collection_unstructured_obj = client.get_collection(
        name=collection_name_unstructured,
        embedding_function=chroma_embedding_function
    )
    collection_docling_obj = client.get_collection(
        name=collection_name_docling,
        embedding_function=chroma_embedding_function
    )
except Exception:
    # Nếu collection chưa tồn tại, tạo mới bằng cách insert document đầu tiên
    collection_unstructured_obj = None
    collection_docling_obj = None

print("\n🚀 BẮT ĐẦU XỬ LÝ (Incremental - chỉ file mới)...\n")

# Tracking statistics
stats = {
    'total_files': len(pdf_files),
    'skipped_files': 0,
    'processed_files': 0,
    'unstructured_pages': 0,
    'docling_pages': 0,
    'unstructured_errors': 0,
    'docling_errors': 0
}

# Lặp qua từng file PDF
for idx, pdf_file in enumerate(pdf_files, 1):
    pdf_path = os.path.join(data_dir, pdf_file)
    
    print(f"\n[{idx}/{len(pdf_files)}] 📄 Đang kiểm tra: {pdf_file}")
    print("-" * 100)
    
    # ============================================================
    # KIỂM TRA FILE ĐÃ XỬ LÝ CHƯA
    # ============================================================
    already_in_unstructured = False
    already_in_docling = False
    
    if collection_unstructured_obj:
        already_in_unstructured = is_file_processed(collection_unstructured_obj, pdf_file)
    
    if collection_docling_obj:
        already_in_docling = is_file_processed(collection_docling_obj, pdf_file)
    
    # Nếu CẢ 2 collection đều đã có file này → BỎ QUA HOÀN TOÀN
    if already_in_unstructured and already_in_docling:
        print(f"  ⏭️ BỎ QUA: File đã được xử lý trong cả 2 collections")
        stats['skipped_files'] += 1
        continue
    
    # Nếu 1 trong 2 thiếu → xử lý lại cả 2 để đảm bảo đồng bộ
    if already_in_unstructured or already_in_docling:
        print(f"  ⚠️ CẢNH BÁO: File chỉ có trong 1 collection, sẽ xử lý lại để đồng bộ")
    
    stats['processed_files'] += 1
    start_time = time.time()

    print(f"\n[{idx}/{len(pdf_files)}] 📄 Đang xử lý: {pdf_file}")
    print("-" * 100)
    

    # ============================================================
    # PHƯƠNG PHÁP 1: DOCLING (chỉ xử lý nếu chưa có)
    # ============================================================
    docling_docs = process_with_docling(pdf_path)
    
    if docling_docs:
        save_to_chromadb(docling_docs, collection_name_docling, pdf_file)
        stats['docling_pages'] += len(docling_docs)
    else:
        stats['docling_errors'] += 1
        print(f"    ❌ LỖI: Không có trang nào được trích xuất từ Docling")
    
    # ============================================================
    # PHƯƠNG PHÁP 2: UNSTRUCTURED (chỉ xử lý nếu chưa có)
    # ============================================================
    unstructured_docs = process_with_unstructured(pdf_path)
    
    if unstructured_docs:
        save_to_chromadb(unstructured_docs, collection_name_unstructured, pdf_file)
        stats['unstructured_pages'] += len(unstructured_docs)
    else:
        stats['unstructured_errors'] += 1
        print(f"    ❌ LỖI: Không có trang nào được trích xuất từ Unstructured")
    
    # Tính thời gian xử lý
    elapsed_time = time.time() - start_time
    
    print(f"\n  ⏱️ Thời gian: {elapsed_time:.2f} giây")
    print(f"  📊 Tổng kết: Unstructured ({len(unstructured_docs)} trang), Docling ({len(docling_docs)} trang)")

print("\n" + "="*100)
print("🎉 HOÀN TẤT XỬ LÝ TẤT CẢ FILE!")
print("="*100)

📁 Tìm thấy 43 file PDF trong thư mục ./data

🚀 BẮT ĐẦU XỬ LÝ (Incremental - chỉ file mới)...


[1/43] 📄 Đang kiểm tra: 001. 10_2016_TT-BGDDT_16230919.pdf
----------------------------------------------------------------------------------------------------
  ⏭️ BỎ QUA: File đã được xử lý trong cả 2 collections

[2/43] 📄 Đang kiểm tra: 10. 136a qd ban hanh quy tac van hoa ung xu cua sv.pdf
----------------------------------------------------------------------------------------------------
  ⏭️ BỎ QUA: File đã được xử lý trong cả 2 collections

[3/43] 📄 Đang kiểm tra: 1238 qd ban hanh quy dinh cac hoc phan ngoai ngu trong chuong trinh dao tao trinh do DH tu khoa 2023.pdf
----------------------------------------------------------------------------------------------------
  ⏭️ BỎ QUA: File đã được xử lý trong cả 2 collections

[4/43] 📄 Đang kiểm tra: 1430 qd sua doi bo sung.pdf
----------------------------------------------------------------------------------------------------
  ⏭️ BỎ QUA: F

[INFO] 2025-12-11 10:14:45,559 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2025-12-11 10:14:45,584 [RapidOCR] download_file.py:60: File exists and is valid: D:\LearnUniversity\Semester11\TieuLuanChuyenNganh\LLM_University_Policy_Chatbot\venv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.onnx
[INFO] 2025-12-11 10:14:45,586 [RapidOCR] main.py:53: Using D:\LearnUniversity\Semester11\TieuLuanChuyenNganh\LLM_University_Policy_Chatbot\venv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.onnx
[INFO] 2025-12-11 10:14:45,684 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2025-12-11 10:14:45,695 [RapidOCR] download_file.py:60: File exists and is valid: D:\LearnUniversity\Semester11\TieuLuanChuyenNganh\LLM_University_Policy_Chatbot\venv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2025-12-11 10:14:45,695 [RapidOCR] main.py:53: Using D:\LearnUniversity\Semester11\TieuLuanChuyenNganh\LLM_University_Policy_Chatbot\venv\

    ✅ Hoàn tất! Tổng 1 trang Docling
    💾 Đã lưu/cập nhật 1 documents vào collection 'raw_docling_collection'
  📄 [Unstructured] Đang xử lý: ThacMacBieuDoGiangDay_2025-2026.pdf...


2025-12-11 10:14:57,062 - INFO - PDF text extraction failed, skip text extraction...
2025-12-11 10:14:57,802 - INFO - Reading PDF for file: ./data\ThacMacBieuDoGiangDay_2025-2026.pdf ...


    ✅ Hoàn tất với strategy='hi_res'! Tổng 1 trang
    💾 Đã lưu/cập nhật 1 documents vào collection 'raw_unstructured_collection'

  ⏱️ Thời gian: 19.19 giây
  📊 Tổng kết: Unstructured (1 trang), Docling (1 trang)

[39/43] 📄 Đang kiểm tra: Thông báo nhận chứng chỉ ngoại ngữ để chuyển điểm, miễn ngoại ngữ đầu ra.pdf
----------------------------------------------------------------------------------------------------
  ⏭️ BỎ QUA: File đã được xử lý trong cả 2 collections

[40/43] 📄 Đang kiểm tra: Thông báo nhận đơn xin thay thế môn học.pdf
----------------------------------------------------------------------------------------------------
  ⏭️ BỎ QUA: File đã được xử lý trong cả 2 collections

[41/43] 📄 Đang kiểm tra: Thông báo nhận đơn xin điểm I - đợt 1 HK1.25-26.pdf
----------------------------------------------------------------------------------------------------
  ⏭️ BỎ QUA: File đã được xử lý trong cả 2 collections

[42/43] 📄 Đang kiểm tra: Thông báo nộp lệ phí làm bằng TN ĐHCQ đợt

### Xóa các document của 1 file bất kỳ

In [63]:
# client = chromadb.PersistentClient(path=db_path)  # dùng biến db_path từ notebook
# coll = client.get_collection(name=collection_name_unstructured, embedding_function=chroma_embedding_function)

# filename = "2930 qd quy dinh chuyen doi diem cac hoc phan ngoai ngu danh cho chuong trinh Dh khong chuyen ngu.pdf"  # thay chính xác tên file trong DB (metadata 'filename')
# res = coll.get(where={"filename": {"$eq": filename}}, limit=10000)
# ids_to_delete = res.get("ids", [])

# print("IDs sẽ xóa:", ids_to_delete)
# if ids_to_delete:
#     coll.delete(ids=ids_to_delete)
#     print(f"✅ Đã xóa {len(ids_to_delete)} documents của file '{filename}' khỏi collection '{collection_name_unstructured}'")
# else:
#     print("⚠️ Không tìm thấy documents với filename đó")

In [64]:
import chromadb

def delete_file_from_collections(collection_names: list, filename: str, confirm: bool = False, db_path: str = None) -> dict:
    """
    Xóa tất cả documents có metadata 'filename' == filename trong nhiều collections.
    Trả về dict tóm tắt kết quả: {collection_name: {'found': n, 'deleted': n, 'error': msg_or_None}}
    Chạy với confirm=True để thực sự xóa.
    """
    db_path = db_path or globals().get("db_path", "./chroma_db")
    client = chromadb.PersistentClient(path=db_path)
    results = {}

    for coll_name in collection_names:
        try:
            # Lấy collection (nếu không tồn tại sẽ ném lỗi)
            try:
                coll = client.get_collection(name=coll_name)
            except Exception as e_get:
                results[coll_name] = {"found": 0, "deleted": 0, "error": f"collection not found: {e_get}"}
                print(f"⚠️ Collection '{coll_name}' không tồn tại: {e_get}")
                continue

            # Tìm tất cả docs có filename
            res = coll.get(where={"filename": {"$eq": filename}}, limit=10000, include=['metadatas', 'documents'])
            ids = res.get("ids", []) or []
            found = len(ids)

            if not ids:
                results[coll_name] = {"found": 0, "deleted": 0, "error": None}
                print(f"ℹ️ [{coll_name}] Không tìm thấy documents cho file: {filename}")
                continue

            print(f"ℹ️ [{coll_name}] Tìm thấy {found} document(s) cho file: {filename}")
            print(f"    Ví dụ metadata: {res.get('metadatas',[{}])[0]}")
            preview = (res.get('documents',[ ''])[0] or '')[:400].replace('\n',' ')
            print(f"    Preview: {preview}...")

            if not confirm:
                print(f"⚠️ Chạy hàm với confirm=True nếu muốn thực sự xóa trong '{coll_name}'.")
                results[coll_name] = {"found": found, "deleted": 0, "error": None}
                continue

            # Thực hiện xóa
            coll.delete(ids=ids)
            print(f"✅ Đã xóa {found} document(s) khỏi collection '{coll_name}'")
            results[coll_name] = {"found": found, "deleted": found, "error": None}

        except Exception as e:
            print(f"❌ Lỗi khi xử lý collection '{coll_name}': {e}")
            results[coll_name] = {"found": 0, "deleted": 0, "error": str(e)}

    return results

# Ví dụ sử dụng (bỏ comment và chạy với confirm=True để xóa thực sự):
# delete_file_from_collections(
#     ["raw_unstructured_collection", "raw_docling_collection", "refined_pages_collection"],
#     "TênFile.pdf",
#     confirm=True
# )

### Xóa 1 trang trong 1 file của 1 collection

In [65]:
# ...existing code...
import chromadb

def delete_page_from_collection(collection_name: str, filename: str, page: int, confirm: bool = False) -> bool:
    """
    Xóa document của 1 trang (metadata 'filename' và 'page') trong collection chỉ định.
    Trả về True nếu đã xóa, False nếu không tìm thấy hoặc không confirm.
    """
    try:
        client = chromadb.PersistentClient(path=db_path)
        coll = client.get_collection(name=collection_name)
    except Exception as e:
        print(f"❌ Lỗi khi kết nối ChromaDB: {e}")
        return False

    # Tìm document theo filename + page
    results = coll.get(
        where={
            "$and": [
                {"filename": {"$eq": filename}},
                {"page": {"$eq": page}}
            ]
        },
        include=['metadatas', 'documents']
    )

    ids = results.get('ids', [])
    if not ids:
        print(f"⚠️ Không tìm thấy trang {page} của file '{filename}' trong collection '{collection_name}'")
        return False

    print(f"ℹ️ Tìm thấy {len(ids)} document(s) - ví dụ metadata: {results.get('metadatas', [])[0]}")
    # Hiển thị preview ngắn
    preview = (results.get('documents', [''])[0] or '')[:400].replace('\n',' ')
    print(f"   Preview: {preview}...")

    if not confirm:
        print("⚠️ Chạy hàm với confirm=True nếu muốn thực sự xóa.")
        return False

    # Thực hiện xóa
    try:
        coll.delete(ids=ids)
        print(f"✅ Đã xóa {len(ids)} document(s) khỏi collection '{collection_name}'")
        return True
    except Exception as e:
        print(f"❌ Lỗi khi xóa: {e}")
        return False
    
print('✅ Đã định nghĩa hàm delete_page_from_collection()')

# Ví dụ sử dụng (bỏ dấu # và chỉnh tên collection / filename / page, chạy với confirm=True để xóa)
# delete_page_from_collection("raw_unstructured_collection", "12_NganhCNKTMoiTruong.pdf", 28, confirm=True)

✅ Đã định nghĩa hàm delete_page_from_collection()


## 6. Thống kê tổng quan

In [66]:
print("\n📊 THỐNG KÊ TỔNG QUAN")
print("="*100)
print(f"Tổng số file PDF: {stats['total_files']}")
print(f"   - File mới xử lý: {stats['processed_files']} files")
print(f"   - File đã có (bỏ qua): {stats['skipped_files']} files")

print(f"\n📚 UNSTRUCTURED:")
print(f"  - Tổng số trang mới thêm: {stats['unstructured_pages']}")
print(f"  - Số file lỗi: {stats['unstructured_errors']}")
print(f"  - Tỷ lệ thành công: {((stats['total_files'] - stats['unstructured_errors']) / stats['total_files'] * 100):.1f}%")

print(f"\n📚 DOCLING:")
print(f"  - Tổng số trang mới thêm: {stats['docling_pages']}")
print(f"  - Số file lỗi: {stats['docling_errors']}")
print(f"  - Tỷ lệ thành công: {((stats['total_files'] - stats['docling_errors']) / stats['total_files'] * 100):.1f}%")

# So sánh
diff = abs(stats['unstructured_pages'] - stats['docling_pages'])
print(f"\n⚖️ CHÊNH LỆCH:")
if diff == 0:
    print(f"  ✅ Cả 2 phương pháp trích xuất được GIỐNG NHAU số trang: {stats['unstructured_pages']}")
else:
    print(f"  ⚠️ Chênh lệch {diff} trang giữa 2 phương pháp")
    print(f"     Có thể do: 1 file lỗi hoặc cách phân trang khác nhau")


📊 THỐNG KÊ TỔNG QUAN
Tổng số file PDF: 43
   - File mới xử lý: 1 files
   - File đã có (bỏ qua): 42 files

📚 UNSTRUCTURED:
  - Tổng số trang mới thêm: 1
  - Số file lỗi: 0
  - Tỷ lệ thành công: 100.0%

📚 DOCLING:
  - Tổng số trang mới thêm: 1
  - Số file lỗi: 0
  - Tỷ lệ thành công: 100.0%

⚖️ CHÊNH LỆCH:
  ✅ Cả 2 phương pháp trích xuất được GIỐNG NHAU số trang: 1


## 7. Verification - So sánh kết quả từ 2 collections

In [67]:
print("\n🔍 KIỂM TRA KẾT QUẢ TRONG CHROMADB")
print("="*100)

try:
    # Kết nối đến ChromaDB
    client = chromadb.PersistentClient(path=db_path)
    
    # Lấy 2 collections - SỬA ĐỔI: dùng chroma_embedding_function thay vì embedding_model
    collection_unstructured = client.get_collection(
        name=collection_name_unstructured,
        embedding_function=chroma_embedding_function  # <-- THAY ĐỔI TẠI ĐÂY
    )
    
    collection_docling = client.get_collection(
        name=collection_name_docling,
        embedding_function=chroma_embedding_function  # <-- THAY ĐỔI TẠI ĐÂY
    )
    
    # Đếm số documents
    count_unstructured = collection_unstructured.count()
    count_docling = collection_docling.count()
    
    print(f"\n📊 Số lượng documents trong ChromaDB:")
    print(f"  - Collection '{collection_name_unstructured}': {count_unstructured} documents")
    print(f"  - Collection '{collection_name_docling}': {count_docling} documents")
    
    if count_unstructured == count_docling:
        print(f"\n  ✅ KHỚP! Cả 2 collections có cùng số documents: {count_unstructured}")
    else:
        diff = abs(count_unstructured - count_docling)
        print(f"\n  ⚠️ CHÊNH LỆCH {diff} documents giữa 2 collections")
    
except Exception as e:
    print(f"❌ LỖI khi kiểm tra ChromaDB: {e}")


🔍 KIỂM TRA KẾT QUẢ TRONG CHROMADB

📊 Số lượng documents trong ChromaDB:
  - Collection 'raw_unstructured_collection': 648 documents
  - Collection 'raw_docling_collection': 648 documents

  ✅ KHỚP! Cả 2 collections có cùng số documents: 648


In [68]:
# ...existing code...
import csv
import math
from collections import defaultdict

# ---------------------------------------------------------
# So sánh pages giữa 2 collections: raw_unstructured_collection vs raw_docling_collection
# Kết quả: in ra file nào, page nào chỉ có ở 1 collection, preview ngắn và lưu CSV
# ---------------------------------------------------------

# KẾT NỐI CHROMADB (dùng biến db_path và chroma_embedding_function từ notebook)
client = chromadb.PersistentClient(path=db_path)

coll_u = client.get_collection(name=collection_name_unstructured, embedding_function=chroma_embedding_function)
coll_d = client.get_collection(name=collection_name_docling, embedding_function=chroma_embedding_function)

# Lấy toàn bộ dữ liệu (IDs, metadatas, documents)
data_u = coll_u.get(limit=100000, include=['metadatas', 'documents'])
data_d = coll_d.get(limit=100000, include=['metadatas', 'documents'])

# Hàm helper chuyển page về int nếu có thể
def _safe_page(p):
    try:
        if isinstance(p, (int, float)) and not math.isnan(p):
            return int(p)
        if isinstance(p, str) and p.strip().isdigit():
            return int(p.strip())
    except Exception:
        pass
    return p

# Build mapping: filename -> set(pages) và filename->page->preview
map_u_pages = defaultdict(set)
map_d_pages = defaultdict(set)
map_u_preview = defaultdict(dict)
map_d_preview = defaultdict(dict)

for i, meta in enumerate(data_u.get('metadatas', [])):
    fname = meta.get('filename') or meta.get('source') or '<unknown>'
    page = _safe_page(meta.get('page'))
    map_u_pages[fname].add(page)
    # preview: ngắn gọn, tránh in quá dài
    doc = (data_u.get('documents', []) or [None]*len(data_u.get('metadatas', [])))[i] or ""
    map_u_preview[fname][page] = doc[:800].replace("\n", " ")

for i, meta in enumerate(data_d.get('metadatas', [])):
    fname = meta.get('filename') or meta.get('source') or '<unknown>'
    page = _safe_page(meta.get('page'))
    map_d_pages[fname].add(page)
    doc = (data_d.get('documents', []) or [None]*len(data_d.get('metadatas', [])))[i] or ""
    map_d_preview[fname][page] = doc[:800].replace("\n", " ")

# Tập hợp tất cả filenames để so sánh
all_filenames = sorted(set(list(map_u_pages.keys()) + list(map_d_pages.keys())))

# Tạo list kết quả chi tiết
diff_rows = []
total_only_u = 0
total_only_d = 0

print(f"🔎 So sánh {len(all_filenames)} files giữa '{collection_name_unstructured}' và '{collection_name_docling}'\n")

for fname in all_filenames:
    pages_u = map_u_pages.get(fname, set())
    pages_d = map_d_pages.get(fname, set())
    only_in_u = sorted(p for p in pages_u - pages_d)
    only_in_d = sorted(p for p in pages_d - pages_u)

    if only_in_u or only_in_d:
        print(f"📁 File: {fname}")
        if only_in_u:
            print(f"   ➖ Chỉ có trong {collection_name_unstructured}: {len(only_in_u)} pages -> {only_in_u[:20]}")
            total_only_u += len(only_in_u)
            for p in only_in_u:
                preview = map_u_preview.get(fname, {}).get(p, "")[:300]
                diff_rows.append({'filename': fname, 'page': p, 'only_in': 'unstructured', 'preview': preview})
        if only_in_d:
            print(f"   ➖ Chỉ có trong {collection_name_docling}: {len(only_in_d)} pages -> {only_in_d[:20]}")
            total_only_d += len(only_in_d)
            for p in only_in_d:
                preview = map_d_preview.get(fname, {}).get(p, "")[:300]
                diff_rows.append({'filename': fname, 'page': p, 'only_in': 'docling', 'preview': preview})
        print("-"*80)

print(f"\n📊 Tổng khác biệt: only_in_{collection_name_unstructured} = {total_only_u}, only_in_{collection_name_docling} = {total_only_d}")
# print(f"📥 Lưu kết quả chi tiết vào './chroma/diffs_missing_pages.csv'")

# Lưu CSV chi tiết
# out_path = "./chroma/diffs_missing_pages.csv"
# with open(out_path, "w", encoding="utf-8", newline="") as f:
#     writer = csv.DictWriter(f, fieldnames=['filename','page','only_in','preview'])
#     writer.writeheader()
#     for r in diff_rows:
#         writer.writerow(r)

print("✅ Hoàn tất. Mở cmt để lưu vào file.csv và xem chi tiết.")

# Nếu cần: in một vài ví dụ để debug nội dung page cụ thể (bỏ comment và chỉnh filename/page nếu muốn)
# example_fname = "Tên_file_cần_kiểm_tra.pdf"
# example_page = 13
# print("Preview unstructured:", map_u_preview.get(example_fname, {}).get(example_page))
# print("Preview docling:", map_d_preview.get(example_fname, {}).get(example_page))
# ...existing code...

🔎 So sánh 43 files giữa 'raw_unstructured_collection' và 'raw_docling_collection'


📊 Tổng khác biệt: only_in_raw_unstructured_collection = 0, only_in_raw_docling_collection = 0
✅ Hoàn tất. Mở cmt để lưu vào file.csv và xem chi tiết.


In [69]:
# ...existing code...
import chromadb

def add_empty_page_to_docling(collection_name: str, filename: str, page: int, source_path: str = None, confirm: bool = False) -> bool:
    """
    Thêm 1 document trang rỗng vào collection (docling-style metadata).
    Trả về True nếu đã thêm, False nếu đã tồn tại hoặc không confirm.
    """
    try:
        client = chromadb.PersistentClient(path=db_path)
        # đảm bảo collection tồn tại và dùng chroma_embedding_function giống phần còn lại
        coll = client.get_or_create_collection(name=collection_name, embedding_function=chroma_embedding_function)
    except Exception as e:
        print(f"❌ Lỗi khi kết nối ChromaDB: {e}")
        return False

    # kiểm tra đã có page đó chưa
    res = coll.get(
        where={
            "$and": [
                {"filename": {"$eq": filename}},
                {"page": {"$eq": page}}
            ]
        },
        include=['metadatas']
    )
    if res.get('metadatas'):
        print(f"⚠️ Đã tồn tại {len(res['metadatas'])} document cho file '{filename}' page {page} trong '{collection_name}' - không thêm nữa")
        return False

    # chuẩn bị metadata giống format docling
    metadata = {
        "source": source_path or filename,
        "filename": filename,
        "page": page,
        "method": "docling"
    }
    doc_id = f"{filename}_page_{page}"

    print(f"ℹ️ Sẽ thêm document rỗng: id={doc_id}, metadata={metadata}")
    if not confirm:
        print("⚠️ Chạy hàm với confirm=True nếu muốn thực sự thêm document.")
        return False

    try:
        # upsert: sẽ thêm mới document rỗng (empty string content)
        coll.upsert(
            ids=[doc_id],
            documents=[""],          # nội dung rỗng
            metadatas=[metadata]
        )
        print(f"✅ Đã thêm trang rỗng cho '{filename}' page {page} vào collection '{collection_name}'")
        return True
    except Exception as e:
        print(f"❌ Lỗi khi upsert: {e}")
        return False

# Ví dụ: (bỏ comment và chạy)
# add_empty_page_to_docling(collection_name_docling, "38_NganhHeThongKyThuatCongTrinhXD.pdf", 31, source_path="./data/38_NganhHeThongKyThuatCongTrinhXD.pdf", confirm=True)
# ...existing code...

## 8. So sánh chi tiết - Xem nội dung Page 1 của 1 file cụ thể

In [87]:
# Chọn file để test (file đầu tiên)
# Chọn trang cần xem:
test_page = 4

if pdf_files:
    test_filename = pdf_files[13-1] # Thay đổi số để chọn file khác nếu cần
    print(f"\n🔍 SO SÁNH CHI TIẾT - File: {test_filename}")
    print("="*100)
    
    try:
        # Tìm trang 1 của file này trong collection Unstructured
        results_unstructured = collection_unstructured.get(
            where={
                "$and": [
                    {"filename": test_filename},
                    {"page": test_page}
                ]
            },
            limit=1
        )
        
        # Tìm trang 1 của file này trong collection Docling
        results_docling = collection_docling.get(
            where={
                "$and": [
                    {"filename": test_filename},
                    {"page": test_page}
                ]
            },
            limit=1
        )

        collection_refined = client.get_collection(
        name="refined_pages_collection",
        embedding_function=chroma_embedding_function  # <-- THAY ĐỔI TẠI ĐÂY
    )

        results_refined = collection_refined.get(
            where={
                "$and": [
                    {"filename": test_filename},
                    {"page": test_page}
                ]
            },
            limit=1
        )
        
        # Hiển thị kết quả
        print(f"\n📄 TRANG {test_page} - UNSTRUCTURED:")
        print("-"*100)
        if results_unstructured['documents']:
            content = results_unstructured['documents'][0]
            metadata = results_unstructured['metadatas'][0] if results_unstructured['metadatas'] else {}
            print(f"Metadata: {metadata}")
            print(f"\nNội dung ({len(content[:400])} ký tự đầu):")
            print(content[:400])
        else:
            print(f"  ⚠️ Không tìm thấy trang {test_page}")
        
        print("\n" + "="*100)
        print(f"\n📄 TRANG {test_page} - DOCLING:")
        print("-"*100)
        if results_docling['documents']:
            content = results_docling['documents'][0]
            metadata = results_docling['metadatas'][0] if results_docling['metadatas'] else {}
            print(f"Metadata: {metadata}")
            print(f"\nNội dung ({len(content[:1400])} ký tự đầu):")
            print(content[:1400])
        else:
            print(f"  ⚠️ Không tìm thấy trang {test_page}")

        print("\n" + "="*100)
        print(f"\n📄 TRANG {test_page} - REFINED:")
        print("-"*100)
        if results_refined['documents']:
            content = results_refined['documents'][0]
            metadata = results_refined['metadatas'][0] if results_refined['metadatas'] else {}
            print(f"Metadata: {metadata}")
            print(f"\nNội dung ({len(content[:580])} ký tự đầu):")
            print(content[:580])
        else:
            print(f"  ⚠️ Không tìm thấy trang {test_page}")
        
    except Exception as e:
        print(f"❌ LỖI khi so sánh: {e}")
else:
    print("⚠️ Không có file PDF để test")


🔍 SO SÁNH CHI TIẾT - File: 2810_Quyết định ban hành Quy định thu học phí năm học 2025-2026_1.pdf

📄 TRANG 4 - UNSTRUCTURED:
----------------------------------------------------------------------------------------------------
Metadata: {'method': 'unstructured', 'filename': '2810_Quyết định ban hành Quy định thu học phí năm học 2025-2026_1.pdf', 'total_elements': 19, 'page': 4, 'source': './data\\2810_Quyết định ban hành Quy định thu học phí năm học 2025-2026_1.pdf'}

Nội dung (400 ký tự đầu):
2

3. Hệ chính quy bậc đại học chương trình đào tạo chất lượng cao (các khóa từ 2023 trở về trước).

Đơn vị: Đồng

Khối ngành công nghệ kỹ thuật Khối ngành khoa học xã hội Niên khóa = sản xuất chế biến @ — quản lý kinh doanh ( Mức HP/ Mức HP / Mức HP/ Mức HP / Học kỳ tín chỉ Học kỳ tín chỉ Xa 16.000.000 853.000 | 15.000.000 857.000 trở về trước Khóa 2023 23.200.000 1.237.000|_ 20.800.000 1.189.000


📄 TRANG 4 - DOCLING:
-----------------------------------------------------------------------------

## 9. Kết luận

✅ **Đã hoàn tất:**
1. Trích xuất RAW content từ tất cả PDF files
2. Lưu vào 2 collections riêng biệt trong ChromaDB
3. Mỗi document tương ứng CHÍNH XÁC 1 trang của PDF gốc

**Bước tiếp theo:**
- Sử dụng LLM để merge/compare/clean nội dung từ 2 phương pháp
- Chunking thông minh hơn (semantic chunking)
- Tạo RAG pipeline hoàn chỉnh

In [71]:
# In ra toàn bộ từng trang trong collection Docling (sắp xếp theo filename -> page)

print("\n📚 XEM TOÀN BỘ DOCUMENT CỦA COLLECTION DOCLING TRONG CHROMADB (CHI TIẾT THEO TRANG)")
print("="*120)
try:
    client = chromadb.PersistentClient(path=db_path)
    collection_docling = client.get_collection(
        name=collection_name_docling,
        embedding_function=chroma_embedding_function
    )

    # Debug: in ra danh sách collections để chắc chắn collection tồn tại
    print("🔎 Collections hiện có trong DB:", [c.name for c in client.list_collections()])

    # LẤY TOÀN BỘ DOCS: dùng get(limit=...) thay vì where={"$exists": "filename"} (không hợp lệ)
    # Nếu cần filter hợp lệ, có thể dùng where={"filename": {"$ne": None}}
    all_docs = collection_docling.get(limit=10000)

    docs = all_docs.get("documents", [])
    metas = all_docs.get("metadatas", [])

    total = len(docs)
    print(f"\nTổng documents: {total}\n")

    # Tạo list và sắp xếp theo filename, page
    items = []
    for i in range(total):
        m = metas[i] if metas else {}
        fname = m.get("filename", "<unknown>")
        page = m.get("page", None)
        try:
            page_sort = int(page)
        except Exception:
            page_sort = page or 0
        items.append((fname, page_sort, docs[i], m))

    items.sort(key=lambda x: (x[0], x[1]))

    for idx, (fname, page_no, content, meta) in enumerate(items, start=1):
        print(f"--- [{idx}/{total}] File: {fname} | Page: {page_no} ---")
        print(f"Metadata: {meta}")
        print(f"Nội dung (độ dài {len(content)}):")
        print(content)
        print("\n" + "-"*120 + "\n")

    print("✅ HOÀN TẤT VIỆC XEM TOÀN BỘ COLLECTION DOCLING")
except Exception as e:
    print(f"❌ LỖI khi xem collection Docling: {e}")


📚 XEM TOÀN BỘ DOCUMENT CỦA COLLECTION DOCLING TRONG CHROMADB (CHI TIẾT THEO TRANG)
🔎 Collections hiện có trong DB: ['academic_regulations_test', 'refined_pages_collection', 'academic_regulations', 'raw_docling_collection', 'raw_unstructured_collection']

Tổng documents: 648

--- [1/648] File: 001. 10_2016_TT-BGDDT_16230919.pdf | Page: 1 ---
Metadata: {'method': 'docling', 'filename': '001. 10_2016_TT-BGDDT_16230919.pdf', 'page': 1, 'source': './data\\001. 10_2016_TT-BGDDT_16230919.pdf'}
Nội dung (độ dài 625):
S610/2016/TT-BGDDT

Ha Noi,ngay 05 thang 4 nam 2016

## THONG TU

## Ban hanh Quy che cong tac sinh vien

## doi voi chuong trinh dao tao dai hoc he chinh quy

Can cu Luat Giao duc dai hoc ngay 18 thang 6 nam 2012;

Can cit Nghi dinh s6 32/2008/ND-CP ngay 19 thang 3 nam 2008cua

Chinhphiquy dinhchircnang,nhiemvu,quyenhanva cocautochirccia B GiaoducvaDaotao;

Can cuNghi dinhso75/2006/ND-CPngay 02thang8nam2006cua www.LuatVietnam.vn

CancirNghidinhs0141/2013/ND-CPngay24/10/2013cuaCh